# AVA Drone Agent with real-time 3D visualizationthrough an Unreal Engine 5 TCP relay


In [1]:
!pip install -q agno dronekit pymavlink future pyyaml pydantic


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = input('OpenRouter API Key: ')

print('API key set.' if os.environ.get('OPENROUTER_API_KEY') else 'No API key!')

API key set.


# Vehicle Connection

Connects to the Mission Planner SITL instance via DroneKit.
Only one script can hold this connection at a time. So if we  re-run this cell, we have to restart the kernel first.

In [3]:
from dronekit import connect, VehicleMode, LocationGlobal, LocationGlobalRelative
from pymavlink import mavutil
import time

CONNECTION_STRING = 'tcp:127.0.0.1:5763'

print('Connecting to vehicle...')
vehicle = connect(CONNECTION_STRING, wait_ready=True, baud=57600, rate=60)
print(f'Connected!')
print(f'  Mode:     {vehicle.mode.name}')
print(f'  Armed:    {vehicle.armed}')
print(f'  Battery:  {vehicle.battery}')
print(f'  GPS:      {vehicle.gps_0}')
print(f'  Alt:      {vehicle.location.global_relative_frame.alt}m')

Connecting to vehicle...
Connected!
  Mode:     GUIDED
  Armed:    True
  Battery:  Battery:voltage=12.6,current=28.11,level=87
  GPS:      GPSInfo:fix=6,num_sat=10
  Alt:      10.0m


## 4. TCP Relay for Unreal Engine 5

This section sets up the real-time telemetry bridge between DroneKit and UE5.

Architecture:
- TCP_Relay runs a background thread that manages a TCP socket on port 1234
- vehicle_to_unreal() converts DroneKit telemetry to UE5-compatible format 
- A telemetry streaming thread reads vehicle state at 60fps and updates relay.message
- The UE5 standalone app connects as a TCP client



In [4]:
import socket
import threading
import math


# --- TCP Relay (from tcp_relay.py) ---

def create_tcp_host(host="127.0.0.1", port=1234, listen=1):
    """Create and return a TCP server socket bound to host:port."""
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.bind((host, port))
    server_socket.listen(listen)
    print(f'TCP relay listening on {host}:{port}')
    return server_socket


def create_fields_string(fields_list):
    """Convert a list of float values to a space-separated string for UE5 parsing."""
    field_str = "{} " * len(fields_list)
    return field_str.format(*fields_list).rstrip()


class TCP_Relay:
    """
    TCP server that relays float data between Python and Unreal Engine 5.
    Runs in a daemon thread so it does not block the main script.

    Attributes:
        message (str):   Outgoing data to UE5. Updated by the telemetry thread.
        message_in (bytes): Incoming data from UE5. Read by the main script if needed.
        linked (bool):   True when UE5 client is connected.
    """
    def __init__(self, num_fields=23, host="127.0.0.1", port=1234, size=1024):
        self.server_socket = create_tcp_host(host, port)
        self.num_fields = num_fields
        self.size = size
        self.message = create_fields_string([0.] * self.num_fields)
        self.linked = False
        self.message_in = None
        self.client_socket = None
        self.thread = threading.Thread(target=self._server)
        self.thread.daemon = True
        self.thread.start()

    def _server(self):
        """Background loop: receive from UE5, then send current message. Reconnects on disconnect."""
        while True:
            while self.linked:
                try:
                    self.message_in = self.client_socket.recv(self.size)
                except socket.error as e:
                    print(f'TCP receive error: {e}')
                    self.message_in = None

                try:
                    self.client_socket.send(str.encode(self.message))
                except socket.error as e:
                    print(f'TCP send error: {e}')
                    print('UE5 client disconnected.')
                    self.client_socket.close()
                    self.linked = False
                    self.client_socket = None

            if not self.linked:
                try:
                    self.client_socket, client_address = self.server_socket.accept()
                    print(f'UE5 client connected: {client_address}')
                    self.linked = True
                except socket.error as e:
                    print(f'TCP accept error: {e}')

            time.sleep(0.002)

    def close(self):
        """Clean shutdown of the relay socket."""
        try:
            if self.client_socket:
                self.client_socket.close()
            self.server_socket.close()
            print('TCP relay closed.')
        except Exception as e:
            print(f'Error closing TCP relay: {e}')



In [5]:
# --- Telemetry Conversion (from dronekit_unreal.py) ---

def vehicle_to_unreal(vehicle, z_invert=True, scale=100):
    """
    Convert DroneKit vehicle telemetry to UE5-compatible values.

    Transformations applied:
        - Local position: meters to centimeters (scale=100)
        - Z axis: inverted (ArduPilot NED 'down-positive' to UE5 'up-positive')
        - Attitude: radians to degrees
        - GPS coordinates: rounded to 8 decimal places
        - Local position: rounded to 3 decimal places (mm precision in cm)
        - Attitude: rounded to 3 decimal places
    """
    d = {}
    d["lat"] = vehicle.location.global_frame.lat
    d["lon"] = vehicle.location.global_frame.lon
    d["alt"] = vehicle.location.global_frame.alt
    d["n"] = vehicle.location.local_frame.north * scale
    d["e"] = vehicle.location.local_frame.east * scale
    d["d"] = vehicle.location.local_frame.down * scale
    if z_invert:
        d["d"] *= -1
    d["roll"] = vehicle.attitude.roll
    d["pitch"] = vehicle.attitude.pitch
    d["yaw"] = vehicle.attitude.yaw

    for k, v in d.items():
        if type(v) == float:
            if k in ["lat", "lon", "alt"]:
                d[k] = round(v, 8)
            elif k in ["n", "e", "d"]:
                d[k] = round(v, 3)
            elif k in ["roll", "pitch", "yaw"]:
                d[k] = round(math.degrees(v), 3)

    return d


### Start the TCP Relay and Telemetry Stream

This cell waits for the vehicle's local frame to become available, then starts:
1. The TCP relay server (listens on port 1234 for UE5)
2. A telemetry streaming thread 

After running this cell, launch the UE5 standalone app. It will connect automatically.

In [6]:
# Wait for local frame data to be available from SITL
print('Waiting for local frame data...')
while not vehicle.location.local_frame.north:
    time.sleep(1)
print('Local frame available.')

# Initialize the TCP relay
relay = TCP_Relay()

# Telemetry streaming flag (used to stop the thread cleanly)
telemetry_active = True


def telemetry_stream():
    """
    Background thread that reads vehicle telemetry and updates the relay message
    at approximately 60fps. Runs until telemetry_active is set to False.

    Field mapping (23 fields):
        [0]  X position (North, cm)
        [1]  Y position (East, cm)
        [2]  Z position (Up, cm)       -- inverted from NED
        [3]  Roll (degrees)
        [4]  Pitch (degrees)
        [5]  Yaw (degrees)
        [6]  Mount0 Roll (degrees)
        [7]  Mount0 Pitch (degrees)
        [8]  Mount0 Yaw (degrees)
        [9]  Mount1 Roll (degrees)
        [10] Mount1 Pitch (degrees)
        [11] Mount1 Yaw (degrees)
        [12] Active camera index (0=Mount0, 1=Mount1)
        [13] Camera FOV (degrees)
        [14-22] Reserved
    """
    while telemetry_active:
        try:
            data = vehicle_to_unreal(vehicle)

            fields = [0.] * relay.num_fields
            fields[0] = data["n"]
            fields[1] = data["e"]
            fields[2] = data["d"]
            fields[3] = data["roll"]
            fields[4] = data["pitch"]
            fields[5] = data["yaw"]
            fields[6] = 0.0   # Mount0 roll
            fields[7] = 0.0   # Mount0 pitch
            fields[8] = 0.0   # Mount0 yaw
            fields[9] = 0.0   # Mount1 roll
            fields[10] = 0.0  # Mount1 pitch
            fields[11] = 0.0  # Mount1 yaw
            fields[12] = 0    # Camera index
            fields[13] = 80.0 # Camera FOV

            relay.message = create_fields_string(fields)
        except Exception as e:
            pass  # Silently continue if vehicle data is temporarily unavailable

        time.sleep(1 / 60)


# Start the telemetry stream in a daemon thread
telemetry_thread = threading.Thread(target=telemetry_stream, daemon=True)
telemetry_thread.start()


print(f'Relay status: listening on port 1234, UE5 connected = {relay.linked}')


Waiting for local frame data...
Local frame available.
TCP relay listening on 127.0.0.1:1234
Relay status: listening on port 1234, UE5 connected = False
UE5 client connected: ('127.0.0.1', 51297)


# Drone Tools



In [7]:
from agno.tools import tool


@tool
def set_mode(mode: str) -> str:
    """
    Set the vehicle flight mode.
    Available modes: GUIDED, ALT_HOLD, RTL, AUTO, STABILIZE, LAND
    Args:
        mode: The flight mode to set (e.g. 'GUIDED', 'RTL')
    """
    vehicle.mode = VehicleMode(mode)
    time.sleep(1)
    return f'Mode set to {vehicle.mode.name}'


@tool
def arm_vehicle(arm: bool) -> str:
    """
    Arm or disarm the vehicle motors.
    Args:
        arm: True to arm, False to disarm
    """
    vehicle.armed = arm
    timeout = 10
    while vehicle.armed != arm and timeout > 0:
        time.sleep(1)
        timeout -= 1
    status = 'armed' if vehicle.armed else 'disarmed'
    return f'Vehicle is now {status}'


@tool
def takeoff(alt: float) -> str:
    """
    Take off to a specific altitude in meters.
    Default altitude is 10 meters if not specified.
    Vehicle must be armed and in GUIDED mode first.
    Args:
        alt: Target altitude in meters
    """
    if not vehicle.armed:
        return 'ERROR: Vehicle must be armed before takeoff'
    if vehicle.mode.name != 'GUIDED':
        return 'ERROR: Vehicle must be in GUIDED mode for takeoff'

    vehicle.simple_takeoff(alt)
    while vehicle.location.global_relative_frame.alt < alt * 0.95:
        current = round(vehicle.location.global_relative_frame.alt, 1)
        print(f'  Climbing... {current}m / {alt}m')
        time.sleep(1)
    return f'Takeoff complete. Altitude: {round(vehicle.location.global_relative_frame.alt, 1)}m'


@tool
def go_to_coords(lat: float, lon: float, alt: float = None, frame: str = 'Relative') -> str:
    """
    Fly to GPS coordinates (latitude, longitude, optional altitude).
    Args:
        lat: Destination latitude in decimal degrees
        lon: Destination longitude in decimal degrees
        alt: Destination altitude in meters. If not provided, maintains current altitude.
        frame: Reference frame - 'Relative' (above home) or 'Global' (absolute). Default: Relative
    """
    if alt is None:
        alt = vehicle.location.global_relative_frame.alt

    if frame == 'Relative':
        dest = LocationGlobalRelative(lat, lon, alt)
    else:
        dest = LocationGlobal(lat, lon, alt)

    vehicle.simple_goto(dest)
    return f'Navigating to lat={lat}, lon={lon}, alt={alt}m ({frame} frame)'


@tool
def go_to_local(x: float, y: float, z: float) -> str:
    """
    Move relative to current position using local frame (Forward/Right/Up).
    Limited to 10 meters per axis for safety.
    Args:
        x: Forward (+) / Backward (-) in meters
        y: Right (+) / Left (-) in meters
        z: Up (+) / Down (-) in meters
    """
    x = max(-10, min(10, x))
    y = max(-10, min(10, y))
    z = max(-10, min(10, z))

    current_lat = vehicle.location.global_relative_frame.lat
    current_lon = vehicle.location.global_relative_frame.lon
    current_alt = vehicle.location.global_relative_frame.alt

    yaw = vehicle.attitude.yaw
    dx_lat = (x * math.cos(yaw) - y * math.sin(yaw)) / 111320.0
    dx_lon = (x * math.sin(yaw) + y * math.cos(yaw)) / (111320.0 * math.cos(math.radians(current_lat)))

    dest = LocationGlobalRelative(
        current_lat + dx_lat,
        current_lon + dx_lon,
        current_alt + z
    )
    vehicle.simple_goto(dest)
    return f'Moving: forward={x}m, right={y}m, up={z}m'


@tool
def set_heading(yaw: float, frame: str = 'Relative') -> str:
    """
    Set the vehicle heading/yaw angle.
    Args:
        yaw: Target yaw angle in degrees.
             Relative frame: -180 to 180 (positive = clockwise, negative = counterclockwise)
             Global frame: 0 to 360 (0=North, 90=East, 180=South, 270=West)
        frame: 'Relative' (turn relative to current heading) or 'Global' (absolute compass heading). Default: Relative
    """
    is_relative = 1 if frame == 'Relative' else 0
    direction = 1

    msg = vehicle.message_factory.command_long_encode(
        0, 0,
        mavutil.mavlink.MAV_CMD_CONDITION_YAW, 0,
        abs(yaw),
        0,
        direction if yaw >= 0 else -1,
        is_relative,
        0, 0, 0
    )
    vehicle.send_mavlink(msg)
    return f'Heading set to {yaw} degrees ({frame})'


@tool
def get_vehicle_status() -> str:
    """
    Get current vehicle telemetry status.
    Returns mode, armed state, battery, altitude, position, speed, and heading.
    Use this when the user asks about the drone's current state.
    """
    heading_deg = round(math.degrees(vehicle.attitude.yaw), 1)
    ue5_status = 'connected' if relay.linked else 'waiting'
    return (
        f'Mode: {vehicle.mode.name} | '
        f'Armed: {vehicle.armed} | '
        f'Battery: {vehicle.battery.voltage}V ({vehicle.battery.level}%) | '
        f'Alt: {round(vehicle.location.global_relative_frame.alt, 1)}m | '
        f'Lat: {vehicle.location.global_relative_frame.lat} | '
        f'Lon: {vehicle.location.global_relative_frame.lon} | '
        f'Groundspeed: {round(vehicle.groundspeed, 1)} m/s | '
        f'Heading: {heading_deg} deg | '
        f'UE5: {ue5_status}'
    )


# Collect all tools
drone_tools = [set_mode, arm_vehicle, takeoff, go_to_coords, go_to_local, set_heading, get_vehicle_status]
print(f'Defined {len(drone_tools)} drone tools:')
for t in drone_tools:
    print(f'  - {t.name}')

Defined 7 drone tools:
  - set_mode
  - arm_vehicle
  - takeoff
  - go_to_coords
  - go_to_local
  - set_heading
  - get_vehicle_status


# Agent Setup

In [8]:
from textwrap import dedent
from agno.agent import Agent
from agno.models.openrouter import OpenRouter
from agno.db.in_memory import InMemoryDb

db = InMemoryDb()
SESSION_ID = 'drone_session_001'

SYSTEM_PROMPT = dedent("""
    You are an AI copilot for drone operations.
    Speak ultra-concisely with an operational tone.

    You have tools to control a drone connected via DroneKit to a Mission Planner SITL.
    The drone's telemetry is being streamed to Unreal Engine 5 for real-time 3D visualization.

    Rules:
    - For flight commands: use the appropriate tool(s)
    - For questions about drone state: use get_vehicle_status
    - For general questions: respond from your knowledge about drone operations
    - If a command requires multiple steps (e.g. 'take off' implies arm + mode + takeoff),
      execute them in the correct sequence
    - Always confirm what you did after executing commands
""")

drone_agent = Agent(
    name='AVA_DroneAgent',
    model=OpenRouter(
        id='qwen/qwen3-vl-235b-a22b-instruct',
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
    instructions=SYSTEM_PROMPT,
    tools=drone_tools,
    db=db,
    session_id=SESSION_ID,
    add_history_to_context=True,
    read_tool_call_history=True,
    retries=3,
    debug_mode=True,
)

print('Agent ready.')
print(f'  Model:   {drone_agent.model.id}')
print(f'  Tools:   {[t.name for t in drone_tools]}')
print(f'  Session: {SESSION_ID}')

Agent ready.
  Model:   qwen/qwen3-vl-235b-a22b-instruct
  Tools:   ['set_mode', 'arm_vehicle', 'takeoff', 'go_to_coords', 'go_to_local', 'set_heading', 'get_vehicle_status']
  Session: drone_session_001


##  Single Command Test

In [9]:
result = drone_agent.run('What is the current status of the drone?')
print('\nAgent Response:', result.content)

DEBUG ***************** Agent ID: ava-droneagent *****************

DEBUG Reading AgentSession: drone_session_001

DEBUG Creating new AgentSession: drone_session_001

DEBUG ** Agent Run Start: 45a61435-609e-4e09-9c3b-e6bf7d75108c ***

DEBUG Processing tools for model

...

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG Created or updated AgentSession record: drone_session_001

DEBUG *** Agent Run End: 45a61435-609e-4e09-9c3b-e6bf7d75108c ****


Agent Response: Drone status: GUIDED mode, armed, 85% battery, 10m altitude, stationary at (-35.363, 149.165), heading -0.6°, UE5 connected.


## Interactive Command Loop

In [10]:
print('AVA Drone Agent -- Interactive Mode')
print('Type commands in natural language. Type "exit" to quit.')
print(f'UE5 relay: {"connected" if relay.linked else "waiting for connection"}\n')

while True:
    user_input = input('\nCommand: ')

    if user_input.lower().strip() == 'exit':
        print('Closing...')
        breakta

    if not user_input.strip():
        continue

    start_time = time.time()

    try:
        result = drone_agent.run(user_input)
        latency = time.time() - start_time

        print(f'\n  Response: {result.content}')
        print(f'  Latency:  {latency:.2f}s')
    except Exception as e:
        print(f'\n  ERROR: {e}')

print('\nSession ended.')

AVA Drone Agent -- Interactive Mode
Type commands in natural language. Type "exit" to quit.
UE5 relay: connected



DEBUG ************** Session ID: drone_session_001 ***************

DEBUG ***************** Agent ID: ava-droneagent *****************

DEBUG Reading AgentSession: drone_session_001

DEBUG ** Agent Run Start: 383d00c5-2d1a-462d-9884-738ab9f5ac9d ***

...

DEBUG Added RunOutput to Agent Session

DEBUG Created or updated AgentSession record: drone_session_001

DEBUG *** Agent Run End: 4b560569-1c75-402e-95d2-cedfcc64ac0b ****


  Response: Moving 10m north. Confirmed.
  Latency:  6.08s
Closing...

Session ended.


TCP receive error: [WinError 10053] An established connection was aborted by the software in your host machine
TCP send error: [WinError 10053] An established connection was aborted by the software in your host machine
UE5 client disconnected.
UE5 client connected: ('127.0.0.1', 51260)
TCP receive error: [WinError 10053] An established connection was aborted by the software in your host machine
TCP send error: [WinError 10053] An established connection was aborted by the software in your host machine
UE5 client disconnected.
UE5 client connected: ('127.0.0.1', 51223)
TCP receive error: [WinError 10053] An established connection was aborted by the software in your host machine
TCP send error: [WinError 10053] An established connection was aborted by the software in your host machine
UE5 client disconnected.
EOF on TCP socket
EOF on TCP socket
EOF on TCP socket
EOF on TCP socket
EOF on TCP socket
EOF on TCP socket
EOF on TCP socket
EOF on TCP socket
EOF on TCP socket
EOF on TCP socket
EO